# Truco SFT Agent GRPO Training on Google Colab

This notebook trains a Qwen-1.5B model to play **Truco Paulista** using Group Relative Policy Optimization (GRPO) on our public dataset `manzoliw/trucobench-sft`.

### Step 1: Clone Repository and Install Dependencies

In [ ]:
# Clone repo to absolute path /content/trucobench if not already cloned
import os
if not os.path.exists("/content/trucobench"):
    !git clone https://github.com/ManzoliW/trucobench.git /content/trucobench

# Change directory to the absolute path to prevent duplication errors on re-run
%cd /content/trucobench

# Install dependencies
!pip install -q trl peft bitsandbytes accelerate datasets transformers

### Step 2: Load Hugging Face Secret and Run GRPO Training

Make sure you have added your Hugging Face write token to Colab's Secrets (key icon in left panel) named **`HF_TOKEN`** and toggled **Notebook access** to ON.

In [ ]:
import os
from google.colab import userdata

# Inject Hugging Face token from Colab Secrets
try:
    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
    print("✅ Hugging Face Token loaded successfully!")
except Exception as e:
    print("❌ Error loading token. Ensure HF_TOKEN secret is configured and access is enabled in the Secrets tab.")

# Execute GRPO training
!python scripts/train-grpo.py \
    --model_id "Qwen/Qwen2.5-1.5B-Instruct" \
    --load_in_4bit \
    --batch_size 1 \
    --gradient_accumulation_steps 8 \
    --group_size 4 \
    --dataset_size 10000 \
    --output_dir "./output/truco-grpo"

### Step 3: Compress and Download Trained Weights

Run the cell below to compress your LoRA adapters and download them to your local computer.

In [ ]:
from google.colab import files
import shutil

# Zip adapter output directory
shutil.make_archive("truco_grpo_adapter", "zip", "./output/truco-grpo")

# Download the zip file
files.download("truco_grpo_adapter.zip")